# Notebook created with Figure code cleaner

In [ ]:
import os
import sys
import pickle
import importlib
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

os.chdir('/project2/yangili1/cfbuenabadn/torino_paper/analysis')
sys.path.append('/project2/yangili1/cfbuenabadn/torino_paper/notebooks/Figure5')

import Figure5_plot_helpers
import Figure5_helpers
importlib.reload(Figure5_plot_helpers)
importlib.reload(Figure5_helpers)

from Figure5_plot_helpers import (
    plot_coverage_panel, plot_loading_boxplot, plot_factor_panel,
    plot_beta_percentage, plot_beta_direction, plot_gsea_dotplot,
    plot_knop1_braak, plot_ir_heatmap, plot_pkfp_counts, plot_twas_summary,
    plot_utr3_changes,
)
from Figure5_helpers import (
    run_all, COVERAGE_PANELS, BOXPLOT_PANELS, FACTOR_PANELS,
)

data_dir = '../notebooks/Figure5/figure_data'
fig_dir  = '../code/results/figures/Fig5_ad'
k3_dir   = '../code/results/figures/Fig5_ad/gao_k3'

In [ ]:
# --- Uncomment to load pre-saved plot-ready data (and skip the next cell) ---
# def L(name):
#     with open(os.path.join(data_dir, f'{name}.pkl'), 'rb') as f:
#         return pickle.load(f)
# iso             = L('iso')
# coverage_panels = L('coverage_panels')
# boxplot_panels  = L('boxplot_panels')
# factor_panels   = L('factor_panels')
# k3_beta_stats   = L('k3_beta_stats')
# gsea_res2d      = L('gsea_res2d')
# knop1_braak     = L('knop1_braak')
# ir_panel_k3     = L('ir_panel_k3')
# pkfp_panels     = L('pkfp_panels')
# twas_panel      = L('twas_panel')
# twas_panel_k3   = L('twas_panel_k3')
# utr3_panel      = L('utr3_panel')

In [ ]:
# Full data pipeline. Sections are independent and expensive, so run_all takes a
# `sections` argument:
#   'panels' per-gene RDS panels (rpy2)   'k3'   beta summary + GSEA + KNOP1
#   'ir'     heatmap layout (pybedtools, slow)
#   'pkfp'   PKFP coverage                'twas' TWAS summary
#   'utr'    3' UTR terminal-exon matrix
# The two slow UTR stages (terminal-exon grouping and the per-rank logistic
# regression) are cached as figure_data/utr_el.pkl and utr_res.pkl and reused
# when present, so 'utr' returns in seconds. Delete those two files to recompute.
data = run_all(data_dir)

iso             = data['iso']
coverage_panels = data['coverage_panels']
boxplot_panels  = data['boxplot_panels']
factor_panels   = data['factor_panels']
k3_beta_stats   = data['k3_beta_stats']
gsea_res2d      = data['gsea_res2d']
knop1_braak     = data['knop1_braak']
ir_panel_k3     = data['ir_panel_k3']
pkfp_panels     = data['pkfp_panels']
twas_panel      = data['twas_panel']
twas_panel_k3   = data['twas_panel_k3']
utr3_panel      = data['utr3_panel']

## KNOP1 / CISD1 example panels — coverage
Source: `analysis/Figure5_Genetics_Plots.ipynb` (cells 33, 83)

In [ ]:
# Source: analysis/Figure5_Genetics_Plots.ipynb (cells 33, 83)
for name, panel in coverage_panels.items():
    plot_coverage_panel(panel)
    # plt.savefig(f'{fig_dir}/{name}.pdf', bbox_inches='tight')
    # plt.savefig(f'{fig_dir}/{name}.png', bbox_inches='tight', dpi=500)
    plt.show()

## Example panels — genotype boxplots
Source: `analysis/Figure5_Genetics_Plots.ipynb` (cells 49, 91)

In [ ]:
# Source: analysis/Figure5_Genetics_Plots.ipynb (cells 49, 91)
for name, (gene, variant, tissue, factor, style, ylabel, xlabel) in BOXPLOT_PANELS.items():
    plot_loading_boxplot(
        boxplot_panels[name], f'{gene}.{factor}',
        ylabel=ylabel, xlabel=xlabel, color=style,
        palette=None if style else ["darkred", "orangered", "gold"])
    # plt.savefig(f'{fig_dir}/{name}.pdf', bbox_inches='tight', dpi=500)
    # plt.savefig(f'{fig_dir}/{name}.png', bbox_inches='tight', dpi=500)
    plt.show()

## Example panels — factor tracks
Source: `analysis/Figure5_Genetics_Plots.ipynb` (cells 134, 86)

`ENSG00000122873.factors` is written by two cells with different `colores`;
cell 86 (the later-executed one, with the explicit red/purple pair) is used.
Its PDF path in the source has a `..factors.pdf` typo, so on disk the PDF sits
under the double-dot name while the PNG uses the single-dot one.

In [ ]:
# Source: analysis/Figure5_Genetics_Plots.ipynb (cells 134, 86)
for name, (gene, factor_list, colores) in FACTOR_PANELS.items():
    plot_factor_panel(factor_panels[name], colores=colores)
    # if gene == 'ENSG00000122873':
    #     plt.savefig(f'{fig_dir}/{gene}..factors.pdf', bbox_inches='tight')
    # plt.savefig(f'{fig_dir}/{name}.pdf', bbox_inches='tight')
    # plt.savefig(f'{fig_dir}/{name}.png', bbox_inches='tight', dpi=500)
    plt.show()

## beta_direction
Source: `analysis/Fig5_AD_figures.ipynb` (cell 73)

Depends on `positive_sig_ir` / `positive_sig_no_ir`, which come from the snmf5 AD
logistic-regression tables (cells 53, 59, 61) rather than the gao_k3 LR output.
They are passed in explicitly so the panel can be redrawn without rerunning that
pipeline; these are the values the published figure is annotated with.

In [ ]:
# Source: analysis/Fig5_AD_figures.ipynb (cell 73)
beta_direction_stats = {'positive_sig_ir': 0.982, 'positive_sig_no_ir': 0.296}

plot_beta_direction(beta_direction_stats)
# plt.savefig(f'{fig_dir}/beta_direction.pdf', bbox_inches='tight')
# plt.savefig(f'{fig_dir}/beta_direction.png', bbox_inches='tight', dpi=500)

## gao_k3/beta_percentage_allIR
Source: `analysis/AD_K3_RetainedIntrons.ipynb` (cell 68; stats from cells 9-12, 30, 57, 58)

In [ ]:
# Source: analysis/AD_K3_RetainedIntrons.ipynb (cell 68)
plot_beta_percentage(k3_beta_stats)
# plt.savefig(f'{k3_dir}/beta_percentage_allIR.pdf', bbox_inches='tight')
# plt.savefig(f'{k3_dir}/beta_percentage_allIR.png', bbox_inches='tight', dpi=500)

## GSEA_AD and gao_k3/GSEA_AD
Source: `analysis/AD_K3_RetainedIntrons.ipynb` (cell 74) and `Fig5_AD_figures.ipynb` (cell 75)

The same enrichment is written to both output directories; the `Fig5_ad/` copy
adds explicit x ticks. Both `savefig` calls are commented out in the sources.

In [ ]:
# Source: analysis/AD_K3_RetainedIntrons.ipynb (cell 74)
plot_gsea_dotplot(gsea_res2d)
# plt.savefig(f'{k3_dir}/GSEA_AD.pdf', bbox_inches='tight')
# plt.savefig(f'{k3_dir}/GSEA_AD.png', bbox_inches='tight', dpi=500)
plt.show()

# Source: analysis/Fig5_AD_figures.ipynb (cell 75) — same data, extra x ticks
plot_gsea_dotplot(gsea_res2d, xticks=[200, 300])
# plt.savefig(f'{fig_dir}/GSEA_AD.pdf', bbox_inches='tight')
# plt.savefig(f'{fig_dir}/GSEA_AD.png', bbox_inches='tight', dpi=500)

## gao_k3/KNOP1_braak_boxplot
Source: `analysis/AD_K3_RetainedIntrons.ipynb` (cell 90)

In [ ]:
# Source: analysis/AD_K3_RetainedIntrons.ipynb (cell 90)
plot_knop1_braak(knop1_braak)
# plt.savefig(f'{k3_dir}/KNOP1_braak_boxplot.pdf', bbox_inches='tight')
# plt.savefig(f'{k3_dir}/KNOP1_braak_boxplot.png', bbox_inches='tight', dpi=500)

## gao_k3/IR_only_progression_labeled.PCC.{0-6}
Source: `analysis/AD_K3_RetainedIntrons.ipynb` (cell 119; layout from cells 75-82)

One gene-body heatmap per braak stage. The source notebook holds seven
near-identical copies of this ~150-line loop differing only in the score column
and output name; this is the PCC one.

In [ ]:
# Source: analysis/AD_K3_RetainedIntrons.ipynb (cell 119)
for i in range(7):
    plot_ir_heatmap(ir_panel_k3, f'braaks_{str(i)}_zloadings.PCC',
                    title=f'braak stage {str(i)}')
    # plt.savefig(f'{k3_dir}/IR_only_progression_labeled.PCC.{str(i)}.pdf',
    #             bbox_inches='tight', dpi=500)
    # plt.savefig(f'{k3_dir}/IR_only_progression_labeled.PCC.{str(i)}.png',
    #             bbox_inches='tight', dpi=500)
    plt.show()

## gao_k3/PKFP_{AC,DLPFC,PCC}_counts
Source: `analysis/Figure5_examples.ipynb` (cells 56, 57, 59)

Three copies differing only in `tissue_id`. The axis title is hard-coded to
'dorsolateral prefrontal cortex' in all three source cells, including the PCC
and AC ones; that is preserved.

In [ ]:
# Source: analysis/Figure5_examples.ipynb (cells 56, 57, 59)
for tissue, panel in pkfp_panels.items():
    plot_pkfp_counts(panel)
    # plt.savefig(f'{k3_dir}/PKFP_{tissue}_counts.pdf', bbox_inches='tight')
    # plt.savefig(f'{k3_dir}/PKFP_{tissue}_counts.png', bbox_inches='tight', dpi=500)
    plt.show()

## TWAS_summary
Source: `analysis/Figure5_summary_plots.ipynb` (cell 41; tables from cells 2, 18, 32, 33)

In [ ]:
# Source: analysis/Figure5_summary_plots.ipynb (cell 41)
plot_twas_summary(twas_panel)
# plt.savefig(f'{fig_dir}/TWAS_summary.pdf', bbox_inches='tight', dpi=500)
# plt.savefig(f'{fig_dir}/TWAS_summary.png', bbox_inches='tight', dpi=500)

## TWAS_summary_K3
Source: `analysis/Figure5_Genetics_K3.ipynb` (cell 39; tables from cells 8, 11, 16, 21, 23-25).

Same plot as `TWAS_summary`, on the K=3 rerun of the genetics: the genome-wide
`gao_k3/sQTL_factor_202609` TWAS and MR tables and the `gao_k3/genetics`
colocboost summary, scored against the gao_k3 isoform bed.

In [ ]:
# Source: analysis/Figure5_Genetics_K3.ipynb (cell 39)
plot_twas_summary(twas_panel_k3)
# plt.savefig(f'{fig_dir}/TWAS_summary_K3.pdf', bbox_inches='tight', dpi=500)
# plt.savefig(f'{fig_dir}/TWAS_summary_K3.png', bbox_inches='tight', dpi=500)

## UTR3_changes
Source: `analysis/AD_UTR.ipynb` (cell 129, via `build_el_matrix_and_plot` in cell 106)

The full upstream chain is reproduced: terminal-exon grouping and the usage
matrix (cells 2/3/5/33/47/70), the per-rank logistic regression (cell 109), the
rank-pattern summary (cell 87) and the extreme-rank selection (cell 106).

The two slow stages are cached (`figure_data/utr_el.pkl`, `utr_res.pkl`) and
reused, so re-running this costs seconds rather than re-fitting the regression.

`AD_UTR.ipynb` defines `logistic_regression_groups` four times (cells 59, 60,
61, 64). The version used is **cell 64, the one taking `min_sample_frac`** —
also the highest-numbered execution, so the definition live when `res` was
computed.

In [ ]:
# Source: analysis/AD_UTR.ipynb (cell 129)
plot_utr3_changes(utr3_panel)
# plt.savefig(f'{fig_dir}/UTR3_changes.pdf', bbox_inches='tight', dpi=500)
# plt.savefig(f'{fig_dir}/UTR3_changes.png', bbox_inches='tight', dpi=500)